# Task 2 — Classification Algorithms
**Course:** Machine Learning & Deep Learning  
**Points:** 10/60  
**School of Artificial Intelligence and Data Science**

---

## Overview
In this task, we explore and compare six established classification algorithms, understand their theoretical foundations, and analyze their strengths and weaknesses.

## 3.1 Theoretical Summary

### 1. k-Nearest Neighbors (k-NN)
k-NN is a non-parametric, distance-based classifier. It classifies a new sample by finding the **k** closest training points (using Euclidean, Manhattan, or Minkowski distance) and assigning the majority class.

- **Key hyperparameters:** `k` (number of neighbors), distance metric, weighting scheme
- **Complexity:** Train O(1), Inference O(n·d) where n=samples, d=dimensions
- **Best use cases:** Small datasets, non-linear boundaries
- **Limitations:** Slow at inference, sensitive to irrelevant features and scale

---

### 2. Decision Trees
Decision Trees recursively partition the feature space using axis-aligned splits selected to minimize impurity (Gini index or Information Gain/Entropy).

$$\text{Gini}(t) = 1 - \sum_{i=1}^{C} p_i^2$$

$$\text{Entropy}(t) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

- **Key hyperparameters:** `max_depth`, `min_samples_split`, `criterion`
- **Complexity:** Train O(n·d·log n), Inference O(depth)
- **Best use cases:** Interpretable models, mixed feature types
- **Limitations:** Prone to overfitting, unstable

---

### 3. Random Forest
Random Forest builds an ensemble of Decision Trees via **bagging** (bootstrap aggregation) and random feature subsets at each split, reducing variance through averaging predictions.

- **Key hyperparameters:** `n_estimators`, `max_features`, `max_depth`
- **Complexity:** Train O(T·n·d·log n), Inference O(T·depth)
- **Best use cases:** High-dimensional data, tabular data, robust predictions
- **Limitations:** Less interpretable, higher memory usage

---

### 4. Support Vector Machine (SVM)
SVM finds the **maximum-margin hyperplane** separating classes. For non-linear problems, it uses the **kernel trick** to map data into higher-dimensional space.

$$\min_{w,b} \frac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w^T x_i + b) \geq 1$$

- **Key hyperparameters:** `C` (regularization), `kernel` (rbf, linear, poly), `gamma`
- **Complexity:** Train O(n²~n³), Inference O(n_sv · d)
- **Best use cases:** High-dimensional spaces, text classification, small datasets
- **Limitations:** Slow on large datasets, sensitive to feature scaling

---

### 5. Logistic Regression
Logistic Regression is a probabilistic linear classifier that models the posterior probability using the **sigmoid function**:

$$P(y=1|x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

Parameters are learned by maximizing the log-likelihood (minimizing cross-entropy loss).

- **Key hyperparameters:** `C` (inverse regularization), `solver`, `penalty` (L1, L2)
- **Complexity:** Train O(n·d·iterations), Inference O(d)
- **Best use cases:** Binary/multi-class, interpretable coefficients, baseline model
- **Limitations:** Assumes linear decision boundary, struggles with complex patterns

---

### 6. Naive Bayes
Naive Bayes applies Bayes' theorem with the **conditional independence** assumption between features:

$$P(y|x) \propto P(y) \prod_{i=1}^{d} P(x_i|y)$$

- **Key hyperparameters:** `var_smoothing` (Gaussian NB), `alpha` (Multinomial NB)
- **Complexity:** Train O(n·d), Inference O(d·C)
- **Best use cases:** Text classification, spam filtering, fast baseline
- **Limitations:** Independence assumption rarely holds, poor probability calibration

## 3.2 Comparison Table

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 4)

data = {
    'Algorithm':         ['k-NN',    'Decision Tree', 'Random Forest', 'SVM',      'Logistic Regression', 'Naive Bayes'],
    'Interpretability':  ['Low',     'High',          'Low',           'Low',      'High',                'Medium'],
    'Scalability':       ['Low',     'Medium',        'Medium',        'Low',      'High',                'High'],
    'Outlier Sensitivity':['High',   'Medium',        'Low',           'Low',      'Medium',              'Medium'],
    'High-Dim Data':     ['Poor',    'Medium',        'Good',          'Excellent','Good',                'Good'],
    'Multi-class':       ['Native',  'Native',        'Native',        'OvR/OvO',  'Native',              'Native'],
    'Training Speed':    ['Instant', 'Fast',          'Moderate',      'Slow',     'Fast',                'Very Fast'],
    'Overfitting Risk':  ['High',    'High',          'Low',           'Low-Med',  'Low',                 'Low'],
}

df = pd.DataFrame(data)
df.set_index('Algorithm', inplace=True)

# Display as styled table
df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#D9E1F2')]},
]).set_caption('Comparison of Six Classification Algorithms')

## 3.3 Algorithm Deep Dive — Support Vector Machine (SVM)

### Mathematical Derivation

**Goal:** Find hyperplane $w^T x + b = 0$ that maximizes the margin $\frac{2}{\|w\|}$ between classes.

**Primal Problem:**
$$\min_{w,b} \frac{1}{2}\|w\|^2 \quad \text{subject to} \quad y_i(w^T x_i + b) \geq 1 \quad \forall i$$

**Lagrangian Dual:**
$$\max_{\alpha} \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j x_i^T x_j$$
$$\text{s.t.} \quad \alpha_i \geq 0, \quad \sum_i \alpha_i y_i = 0$$

**Soft-Margin (C-SVM):**
$$\min_{w,b,\xi} \frac{1}{2}\|w\|^2 + C\sum_i \xi_i \quad \text{s.t.} \quad y_i(w^T x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

**Kernel Trick:**
Replace $x_i^T x_j$ with $K(x_i, x_j) = \phi(x_i)^T \phi(x_j)$. Common kernels:
- RBF: $K(x,z) = \exp(-\gamma\|x-z\|^2)$
- Polynomial: $K(x,z) = (x^T z + c)^d$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_classification, make_circles

# ─── Hand-crafted 2D example ───────────────────────────────────────────────
np.random.seed(42)
X_simple = np.array([[1,2],[2,3],[3,3],[2,1],[3,1],   # Class 0
                      [5,5],[6,5],[5,6],[6,6],[7,5]])   # Class 1
y_simple = np.array([0,0,0,0,0, 1,1,1,1,1])

# ─── SVM with linear kernel ────────────────────────────────────────────────
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_simple, y_simple)

# ─── RBF kernel on non-linear data ────────────────────────────────────────
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_circles, y_circles)

def plot_decision_boundary(ax, model, X, y, title):
    h = 0.02
    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolors='k', s=60)
    if hasattr(model, 'support_vectors_'):
        ax.scatter(model.support_vectors_[:,0], model.support_vectors_[:,1],
                   s=150, facecolors='none', edgecolors='black', linewidths=2, label='Support Vectors')
        ax.legend(fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(axes[0], svm_lin, X_simple, y_simple, 'SVM — Linear Kernel (Hand-crafted Example)')
plot_decision_boundary(axes[1], svm_rbf, X_circles, y_circles, 'SVM — RBF Kernel (Non-linear Data)')
plt.tight_layout()
plt.savefig('T1_SVM_decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# ─── Manual calculation example ────────────────────────────────────────────
print('=== Manual SVM Calculation Example ===')
print()
print('Dataset (2D, linearly separable):')
print('Class +1: (1,2), (2,3), (3,3)')
print('Class -1: (2,1), (3,1), (1,0)')
print()
print('Step 1: Find support vectors (points closest to boundary)')
print('Step 2: Compute margin = 2 / ||w||')
print('Step 3: Maximize margin → minimize ||w||²/2')
print()

X_manual = np.array([[1,2],[2,3],[3,3],[2,1],[3,1],[1,0]], dtype=float)
y_manual = np.array([1,1,1,-1,-1,-1])

svm_manual = SVC(kernel='linear', C=1e6)  # Hard margin
svm_manual.fit(X_manual, y_manual)

w = svm_manual.coef_[0]
b = svm_manual.intercept_[0]
margin = 2 / np.linalg.norm(w)

print(f'Learned weights:   w = [{w[0]:.4f}, {w[1]:.4f}]')
print(f'Learned bias:      b = {b:.4f}')
print(f'Margin:            {margin:.4f}')
print(f'Support vectors:\n{svm_manual.support_vectors_}')
print()
print('Overfitting: high C → hard margin → memorizes training data')
print('Underfitting: low C → soft margin → may misclassify training points')

## Summary

| Algorithm | Best For | Key Weakness |
|---|---|---|
| k-NN | Small datasets, non-linear | Slow inference, scale-sensitive |
| Decision Tree | Interpretability | Overfitting |
| Random Forest | General purpose, robust | Memory, less interpretable |
| SVM | High-dim, small-medium data | Slow on large n |
| Logistic Regression | Linear problems, baseline | Linear boundary only |
| Naive Bayes | Text, fast baseline | Independence assumption |

**Conclusion:** No single algorithm dominates all scenarios. The choice depends on dataset size, dimensionality, interpretability requirements, and computational budget.